# Carga de los datos para su procesamiento

In [37]:
!pip install wfdb

In [38]:
!rm cargadatos.py

In [39]:
from google.colab import files
uploaded = files.upload()

Saving cargadatos.py to cargadatos.py


In [40]:
from cargadatos import carga_archivos_entorno, proceso_interno_carga

# Procesamiento de los datos para trabajar con ellos

In [41]:
# Importación de las bibliotecas necesarias

import numpy as np
import os

from scipy.signal import butter, filtfilt

Lo primero que vamos a hacer es extraer la señal monocanal o de una derivada que conforma nuestro ECG, ya que nuestro dataset se encuentra formado únicamente por señales de este tipo.

Esto es necesario porque el objeto wfdb.Record que tenemos presenta un montón de información acerca de esta señal, sin embargo, lo que a nosotros nos interesa para la red GAN que vamos a entrenar son los valores numéricos de la señal, la matriz de NumPy que los recoge.

Además, como la señal es unidimensional podemos trabajar con la señal aplanada, es decir, como un array 1D, lo cual facilita el trabajo.


In [42]:
# Vamos a cargar un conjunto de ECGs de ejemplo, para comprobar que las funciones implementadas funcionan correctamente

In [43]:
diccionario_señales = carga_archivos_entorno("/content/drive/MyDrive/dataset_ECG/challenge2017/entrenamiento/A00", ["A00005.hea", "A00034.hea", "A00776.hea"])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [44]:
print (diccionario_señales.keys())

dict_keys(['A00005', 'A00034', 'A00776'])


## Obtención de la matriz de Numpy con los datos numéricos de la señal

In [45]:
def obtener_señal_ecg (record):

  """
  Esta función extrae la señal del objeto wfdb.Record, es decir, la matriz de NumPy que contiene los valores numéricos de la señal, que son los que realmente necesitamos para
  poder trabajar con ellos.

  Parámetros
  --------------
    - record(wfdb.Record): Objeto que contiene el objeto de tipo wfdb.Record de uno de los registros de nuestro dataset, el que indiquemos en este caso.

  Return
  --------------
    - señal unidimensional (np.array): Señal 1D extraída a partir del objeto wfdb.Record (matriz) aplanada y en formato de array de NumPy.

  Raises
  --------------
    - ValueError: Si el objeto de tipo wfdb.Record no contiene el atributo 'p_signal' o este es None se muestra por pantalla un mensaje de error.

  """

  if record.p_signal is not None:                           # Comprueba que el objeto wfdb.Record no sea nulo para extraer los valores numéricos de la señal
    señal = record.p_signal                                 # Se crea una variable interna con la matriz de NumPy que recoge los valores numéricos de la señal
    señal_unidimensional = señal.flatten()                  # Aplana la señal, convirtiéndola en un array de NumPy. Esto solo es posible para señales unidimensionales, como las nuestras.
    return señal_unidimensional                             # Devuelve los valores numéricos de la señal al usuario

  else:                                                     # Muestra un mensaje de error cuando no es posible acceder al atributo 'p_signal' porque nuestro Record no lo tiene
    raise ValueError("El objeto de tipo wfdb.Record no contiene el atributo 'p_signal' ")



In [46]:
for nombre, record in diccionario_señales.items():
  señal = obtener_señal_ecg(record)
  print (señal)
  print (len(señal))

[-0.188 -0.239 -0.274 ... -0.093 -0.057  0.   ]
18000
[ 0.092  0.124  0.162 ... -0.298 -0.18  -0.044]
9000
[0.017 0.017 0.017 ... 0.013 0.013 0.014]
9000


## Filtrado de la señal dentro de un rango de frecuencias

A continuación vamos a aplicar un filtro pasabanda a la señal para quedarnos con los rangos de frecuencia que nos interesan. Esto es importante sobre todo en señales del cuerpo humano para evitar aquellas frecuencias que pueden pertenecer a otro tipo de sonidos que no son ECGs. De esta forma eliminaremos los sonidos de baja y alta frecuencia respectivamente, los cuales pueden influir negativamente en la interpretación o entrenamiento de la red GAN, limpiando así nuestra señal.

- Sonidos de baja frecuencia (por debajo de 0.5): Movimiento del paciente (baseline wander o “deriva de línea base”), interferencia del contacto de electrodos, fluctuaciones del sensor.

- Sonidos de alta frecuencia (por encima de 40): Actividad muscular (EMG), ruido eléctrico (como interferencia a 50/60 Hz), artefactos del entorno.

In [47]:
def filtro_pasa_banda (record, lim_inf=0.5, lim_sup=40, orden=4):

  """
  La función aplica un filtro pasa banda de tipo Butterworth a la señal, para quedarse con las frecuencias deseadas, evitando así que el ruido u otros sonidos corporales puedan
  interferir en el resultado del entrenamiento. Es decir, filtra la señal dentro del rango de frecuencias deseado.

  Parámetros
  ------------
    - record (wfdb.Record): Objeto de tipo Record de uno de los registros de nuestro dataset, el cual contiene la señal a filtrar y la frecuencia de muestreo en el método '.fs'
    - lim_inf (float, optional): Frecuencia de corte inferior de la señal ECG. Si no indicamos nada, es 0.5 Hz por defecto
    - lim_sup (float, optional): Frecuencia de corte alta de la señal de ECG del objeto wfdb.Record. Tiene un valor por defecto de 40 Hz.
    - orden (int, optional): Indica el orden del filtro de Butterworth. Es 4 por defecto, a no ser que indiquemos otro valor para dicho filtro.



  Return
  ----------
    - señal_filtrada (np.array): Array de NumPy con la señal filtrada dentro de los rangos de frecuencia pasados como argumento a la señal.


  Raises
  ----------
    - ValueError: Mensaje de error que indica que los límites de filtrado no se encuentran dentro del rango válido para la frecuencia de Nyquist.


  """


  frec_ecg = record.fs                                           # Obtiene la frecuencia de muestreo a la que se ha grabado la señal de ECG
  señal_ecg_1d = obtener_señal_ecg(record)                       # Consigue el array de NumPy que contine la señal del objeto 'record' aplanada
  frec_nyquist = 0.5 * frec_ecg                                  # Calcula la frecuencia de Nyquist, lo cual es vital para evitar el proceso de 'aliasing'

  if lim_inf > 0 and lim_sup < frec_nyquist:                     # Comprueba si los límites (inferior y superior) cumplen con la frecuencia de Nyquist, para que el filtrado sea útil
    lim_inf_normal = lim_inf / frec_nyquist                      # Normaliza la frecuencia del límite inferior con respecto a la frecuencia de Nyquist, para poder aplicar el filtro
    lim_sup_normal = lim_sup / frec_nyquist                      # Normaliza la frecuencia del límite superior con respecto a la frecuencia de Nyquist, necesario para la función 'butter'

    coef_b, coef_a = butter(orden, [lim_inf_normal, lim_sup_normal], btype='band')     # Calcula los coeficientes necesarios para el filtrado en base a los argumentos pasados
    señal_filtrada = filtfilt(coef_b, coef_a, señal_ecg_1d)                            # Realiza el filtrado de la función mediante un filtro pasa banda de Butterworth

  else:                                                          # Se muestra un mensaje de error por pantalla cuando los coeficientes no cumplen con la frecuencia de Nyquist
    raise ValueError("Los límites del filtro deben estar dentro del rango de Nyquist.")

  return señal_filtrada                                          # Devuelve la señal (array de NumPy) una vez que ha sido filtrada dentro de los límites indicados en el argumento


In [48]:
lista_señales_filtradas = []
for nombre, record in diccionario_señales.items():
  señal_filtrada = filtro_pasa_banda (record)
  lista_señales_filtradas.append(señal_filtrada)
  print (señal_filtrada)
  print (len(señal_filtrada))
print (lista_señales_filtradas)
print (len(lista_señales_filtradas))

[-0.05855155 -0.10600634 -0.15032134 ... -0.05256609 -0.00117434
  0.05004227]
18000
[-0.00038538  0.03645205  0.07420487 ... -0.15749075 -0.05172462
  0.07617379]
9000
[0.00578124 0.00585131 0.00597664 ... 0.00318729 0.00397518 0.0047576 ]
9000
[array([-0.05855155, -0.10600634, -0.15032134, ..., -0.05256609,
       -0.00117434,  0.05004227]), array([-0.00038538,  0.03645205,  0.07420487, ..., -0.15749075,
       -0.05172462,  0.07617379]), array([0.00578124, 0.00585131, 0.00597664, ..., 0.00318729, 0.00397518,
       0.0047576 ])]
3


## Segmentación de la señal con solapamiento de ventanas

Nuestros ECG son un tipo de datos fisiológicos, por lo tanto, es recomendable trabajar con señales segmentadas y con solapamiento de ventanas, tal y como vamos a ver a continuación.

Si por ejemplo nuestra señal tiene 18.000 muestras quiere decir que se han tomado los valores discretos de esa función en 18.000 puntos distintos del ECG.

Sin embargo, para entrenar una red GAN con datos de este tipo es mucho más recomendable hacerlo con datos de longitud pequeña y fija, es decir, por ejemplo de un total de 300 muestras o puntos. Ese es el motivo por el que vamos a realizar la segmentación de la señal en ventanas o segmentos de la misma.
Esto a su vez actúa como una especie de Data Augmentation, ya que vamos a tener una mucho mayor cantidad de señales que pasarle a la red GAN para su entrenamiento.

Además, vamos a hacer un solapamiento, para evitar perdernos algún evento importante de la señal, ya que justo puede que nuestro segmento acabe con un evento de este tipo. Es por esto por lo que vamos a utilizar el solapamiento, teniendo en varios segmentos los mismos eventos con un desplazamiento, para poder evitar esta posible pérdida de información.



In [49]:
for nombre, record in diccionario_señales.items():
  print (f"La frecuencia de muestreo de nuestros ECG es: {record.fs}")

La frecuencia de muestreo de nuestros ECG es: 300
La frecuencia de muestreo de nuestros ECG es: 300
La frecuencia de muestreo de nuestros ECG es: 300


In [50]:
def segmentacion_señal_solapamiento (señal, tamaño_ventana, paso_solapamiento):

  """
  Esta función segmenta la señal unidimensional procedente de un ECG en fragmentos del mismo tamaño para con ellos entrenar una red GAN utilizando la biblioteca de NumPy para ello.
  Para ello se emplea desplazamiento con solapamiento, para extraer de esta forma varias ventanas consecutivas de tamaño fijo y no perder información acerca de la señal.
  La distancia entre las ventanas viene definida por el parámetro 'paso_solapamiento' que se pasa a la función como argumento.

  Para que esta segmentación pueda tener lugar es necesario tener en cuenta que el tamaño de la ventana debe ser menor que la longitud de la propia señal, ya que lo contrario
  no es viable, porque no llegaríamos ni a poder obtener la primera ventana.

  Parámetros
  -------------
    - señal (np.array): Señal unidimensional en forma de array de NumPy que contiene los valores numéricos de un objeto tipo wfdb.Record, la cual va a ser segmentada
    - tamaño_ventana (int): Valor numérico que indica la longitud de valores numéricos (muestras) que va a presentar cada una de las ventanas en las que se divide la señal
    - paso_solapamiento (int): Número de muestras que va a haber entre el inicio de una ventana y la siguiente, es decir, valores que se desplaza la ventana en cada iteración


  Return
  -----------
    - segmentos_solapamiento (np.array): Array bidimensional (2D) donde cada una de las filas corresponde a uno de los segmentos de la señal. Este array presenta la siguiente
    forma --> [número_segmentos, tamaño_ventana]

  Raises
  -----------
    - ValueError: Se muestra un error por pantalla cuando la longitud de la ventana es mayor que el propio tamaño de la señal, ya que no es posible realizar la segmentación
  """

  señal_array = np.asarray (señal)                              # Convierte la señal a array 1D (aunque ya lo debería ser) para asegurarse de que es posible la segmentación

  if tamaño_ventana > len(señal_array):                         # Comprueba que el tamaño de la ventana no sea mayor que el de la propia señal, y si lo es, muestra un error
    raise ValueError ("El tamaño de la ventana es mayor que la propia longitud de la señal, por lo que es imposible llevar a cabo la segmentación.")

  ventanas = np.lib.stride_tricks.sliding_window_view(señal_array, window_shape=tamaño_ventana)       # Crea una vista de las ventanas para la división de las muestras de la señal

  segmentos_solapamiento = ventanas[::paso_solapamiento]        # Crea un array bidimensional en el que almacena cada uno de los segmentos en función del 'paso_solapamiento'


  return segmentos_solapamiento                                 # Devuelve el array que ha creado de la forma [num_segmentos, tamaño_ventana] con los segmentos en que divide la señal


In [51]:
for i in lista_señales_filtradas:
  segmentos = segmentacion_señal_solapamiento (i, 300, 100)
  print(len(segmentos))


178
88
88


## Normalización de los segmentos de la señal

Es necesario llevar a cabo un proceso de normalización de los segmentos obtenidos a partir de las señales del dataset por los siguientes motivos:

- Estabilidad y velocidad de entrenamiento
La normalización hace que los valores de entrada estén en un rango similar (por ejemplo, [−1,1] o [0,1]). Esto evita que la red tenga que aprender a lidiar con rangos de datos muy amplios o con valores muy grandes o muy pequeños, lo que ayuda a que el entrenamiento sea más estable y converja más rápido.

- Evitar sesgo por magnitudes diferentes
Sin normalización, segmentos con amplitudes más grandes podrían dominar el aprendizaje, y la red podría ignorar patrones importantes en señales con amplitudes menores.

- Mejor rendimiento en redes neuronales
Muchas funciones de activación (como tanh, sigmoid, ReLU) funcionan mejor cuando los datos de entrada están escalados adecuadamente.

- Facilita la comparación entre señales
Si diferentes segmentos vienen de distintos pacientes o condiciones, normalizarlos ayuda a que el modelo vea todos los datos "en igualdad de condiciones", centrando la atención en las formas y patrones, no en la escala absoluta.

- Consistencia con datos sintéticos
En GANs, cuando generas señales sintéticas, también quieres que estén en el mismo rango de valores que los datos reales para que el discriminador no distinga fácilmente señales por escala.



In [52]:
# Es recomendable utilizar una media y desviación típica común para todo el dataset, por las ventajas que puede tener esto

#Por ello, es recomendable calcular estos parámetros en un primer momento

In [53]:
def obtener_param_globales (diccionario_dataset):

  """
  La función calcula tanto la media como la desviación estándar global para el dataset completo con el que vamos a trabajar.
  Esto es importante para llevar a cabo la normalización de las señales, presentando así todos los registros del diccionario la misma escala, mejorando así la consistencia
  estadística y la convergencia del modelo.

  Parámetros
  -------------
    - diccionario_dataset (dict): Diccionario en el que se encuentran todos los registros (objetos wfdb.Record) que conforman el dataset con el que vamos a trabajar


  Return
  ---------
    - media_global (float): Valor medio de las señales que componen el dataset con el que vamos a trabajar
    - desviacion_global (float): Desviación típica de los registros que conforman el set de datos que utilizamos


  Raises
  ----------
    - ValueError: Muestra un error por pantalla si el diccionario con los registros pasado como argumento no es realmente un diccionario o se encuentra vacío

  """


  if not diccionario_dataset:                                        # Muestra un error por pantalla si el diccionario pasado como argumento se encuentra vacío
    raise ValueError ("El diccionario está vacío. Inténtalo de nuevo.")

  lista_señales_1d = []                                              # Crea una lista en la que se van a almacenar las señales 1D de cada uno de los registros del diccionario
  for nom, record in diccionario_dataset.items():                    # Recorre cada uno de los registros / entradas del diccionario
    señal = obtener_señal_ecg(record)                                # Obtiene los valores numéricos de cada record con la función 'obtener_señal_ecg()' en un array de NumPy
    lista_señales_1d.append (señal)                                  # Añade este array a la lista creada para almacenar este tipo de dato

  señales_concatenadas = np.concatenate (lista_señales_1d)           # Concatena todos los valores numéricos de la lista, para calcular la media y la desviación de todos


  media_global = np.mean (señales_concatenadas)                      # Calcula la media global para todo el conjunto de registros del dataset
  desviacion_global = np.std (señales_concatenadas)                  # Calcula la desviación estándar del conjunto de datos del dataset

  return media_global, desviacion_global                             # Devuelve la media y desviación global del dataset, las cuales vamos a utilizar para el proceso de normalización




In [54]:
media_dataset, desviacion_dataset = obtener_param_globales (diccionario_señales)

print (media_dataset)
print (desviacion_dataset)

0.01608041666666667
0.27215536740138335


In [55]:
def normalizar_señal (segmento, media_global, desviacion_global):

  """
  Esta función lleva a cabo el proceso de normalización de uno de los segmentos de tamaño fijo en que se divide una señal, utilizando para ello la media y la desviación global del
  dataset en el que se encuentra dicha señal.
  Es importante la universalidad de estos parámetros en el dataset para que todas las señales, y los segmentos en que se dividen estas, se encuentren en la misma escala, y poder
  así trabajar con ellas de forma más eficiente.

  Parámetros
  -------------
    - segmento (np.array): Segmento de tamaño fijo en el que se ha dividido la señal, y el cual va a ser normalizado
    - media (float): Media global del dataset con el que vamos a trabajar
    - desviacion (float): Desviación típica global para el set de datos al que pertenece el segmento a normalizar


  Return
  ---------
    - segmento_normalizado (np.array): Segmento de tamaño fijo, resultado de dividir la señal, pero ya normalizado según los parámetros globales del dataset



  Raises
  ---------
    - ValueError: Muestra un error por pantalla si la desviación estándar es igual a cero, ya que esto quiere decir que no es posible llevar a cabo la normalización del segmento
  """

  if desviacion_global != 0:                                                # Comprueba que la desviación de la señal sea distinta de cero, para poder normalizar la señal
    segmento_normalizado = (segmento - media_global) / desviacion_global    # Realiza el proceso de normalización de la señal
    return segmento_normalizado                                             # Devuelve el segmento concreto con el que estamos trabajando ya normalizado

  else:                                                                     # Muestra un error por pantalla si la desviación típica es igual a 0
    raise ValueError ("Para poder realizar el proceso de normalización la desviación debe ser distinta de 0. Inténtalo de nuevo.")

In [56]:
for i in segmentos:
  segm_norm = normalizar_señal (i, media_dataset, desviacion_dataset)

# Pipeline común para el procesamiento de los datos

Una vez que hemos definido de forma individual todas las funciones que van a realizar el procesamiento de nuestra señal, podemos crear un PipeLine en forma de función que realice todas estas pequeñas modificaciones necesarias y hacer así que los datos lleven a cabo el procesamiento en la totalidad de su conjunto, sin tener que realizar la llamada a cada una de las funciones individualmente.

In [57]:
def procesar_record_individual (record, muestras_ventana, paso_ventana, mean_global, std_global, corte_bajo = 0.5, corte_alto = 40, ord = 4):

  """
  Esta función combina todas las funciones previamente definidas, con el objetivo de llevar a cabo un preprocesamiento de la señal de un record individual de un dataset de ECG y
  poder entrenar una red GAN con señales de este tipo. Para ello, se obtiene la señal (valores numéricos), se aplica un filtro pasa banda, se segmenta en ventanas de un tamaño
  fijo y se normaliza en base a la media y a la desviación típica global del dataset.


  Parámetros
  --------------
    - record (wfdb.Record): Objeto de tipo wfdb.Record que equivale a uno de nuestros registros del dataset, y el cual contiene la señal de ECG y la frecuencia de muestreo de la misma
    - muestras_ventana (int): Número de muestras, o mediciones del valor de la señal que va a contener cada uno de los segmentos en los que se va a dividir la señal
    - paso_ventana (int): Desviación de la ventana para cada iteración, es decir, número de muestras que se desplaza entre un segmento o ventana y el siguiente
    - mean_global (float): Media universal de todos los valores de las señales que conforman la totalidad del dataset. Esta se calcula para la totalidad del dataset
    - std_global (float): Desviación típica calculada a partir de todas las señales que contiene el set de datos de trabajo. Esta se calcula en base a todo el dataset
    - corte_bajo (float, optional): Frecuencia mínima (Hz) que se va a mantener como parte de la señal en el proceso de filtado pasa banda que se va a llevar a cabo
    - corte_alto (float, optional): Frecuencia máxima (Hz) que el filtrado pasa banda va a permitir conservar como valor numérico de la señal procesada
    - ord (int, optional): Orden del filtro Butterworth aplicado como parte del filtro pasa banda

  Return
  --------------
    - segmentos_procesados (list): Lista que contiene los arrays de NumPy que almacenan la señal procesada en cada uno de los segmentos generados
   """

  segmentos_procesados = []                                       #Inicialización de la lista en la que se van a almacenar los arrays de cada uno de los segmentos una vez procesados
  señal_ecg = obtener_señal_ecg (record)                          #Extracción de la señal del objeto wfdb.Record pasado como argumento a la función
  señal_filtrada = filtro_pasa_banda (record, lim_inf=corte_bajo, lim_sup =corte_alto, orden = ord)    #Aplicación del filtro pasa banda a la señal que ha sido extraída
  segmentos = segmentacion_señal_solapamiento (señal_filtrada, muestras_ventana, paso_ventana)         # Segmentación con solapamiento sobre la señal filtrada

  for i in segmentos:                                             # Recorrido cada uno de los valores / muestras que componen los segmentos en que se ha dividido la señal
    segm_proc = normalizar_señal (i, mean_global, std_global)     # Normalización de cada uno de los segmentos en base a la media y desviación global del dataset
    segmentos_procesados.append (segm_proc)                       # Adición del segmento normalizado a la lista vacía que se ha creado para su almacenamiento

  return segmentos_procesados                                     # Devolución de todos los segmentos de la señal filtrados y normalizados



In [58]:
for nom, record in diccionario_señales.items():
  segmentos_procesados = procesar_record_individual (record, 300, 100, media_dataset, desviacion_dataset)
  print (segmentos_procesados[0])

[-0.2742256  -0.44859214 -0.61142191 -0.75183176 -0.86124674 -0.93580965
 -0.97802741 -0.99601189 -1.00029195 -0.99998702 -1.00040932 -1.00292452
 -1.00646449 -1.00945333 -1.01113273 -1.01189843 -1.01283768 -1.01495911
 -1.01860113 -1.02328634 -1.02800745 -1.03172941 -1.03384433 -1.03439518
 -1.03402164 -1.03370056 -1.03440262 -1.03677899 -1.04098942 -1.04677908
 -1.0538223  -1.06216771 -1.07248103 -1.08583879 -1.10310214 -1.12424846
 -1.14817806 -1.17323983 -1.1981729  -1.22280088 -1.24798138 -1.2748601
 -1.30393981 -1.33453476 -1.36489826 -1.39292877 -1.41706642 -1.43688396
 -1.45304    -1.46666592 -1.47862535 -1.48911051 -1.49773224 -1.5039185
 -1.50729816 -1.50782083 -1.50555168 -1.50030097 -1.49140959 -1.47797829
 -1.45951743 -1.43649517 -1.40986075 -1.37874269 -1.33652485 -1.2672093
 -1.14529594 -0.94186256 -0.63656341 -0.2311213   0.24281381  0.7272949
  1.15338105  1.46126583  1.61515408  1.60647729  1.44712656  1.16039449
  0.77663401  0.33456827 -0.11689602 -0.52282914 -0.834

In [59]:
def procesar_multiples_registros (diccionario_registros, tam_ventana, paso_solap, lim_inf = 0.5, lim_sup=40, ord = 4):

  """
  La función se encarga de realizar el procesamiento de las señales de la totalidad de registros de un dataset. Este se pasa como argumento a la función en forma de diccionario.
  Para llevar a cabo esta modificación se va a realizar la modificación individual de cada uno de estos registros, obteniendo su señal, filtrándolos, realizando una segmentación
  con solapamiento y normalizando los mismos con la media y la desviación típica del dataset de trabajo.

  Este procesamiento es esencial para preparar las señales de ECG, como es nuestro caso, para poder entrenar modelos de aprendizaje automático, como son las redes GAN, ya que
  este proceso va a mejorar su eficacia y robustez.

  Parámetros
  ---------------
    - diccionario_registros (dict): Diccionario que contiene todos los obejtos de tipo wfdb.Record que conforman el dataset de trabajo
    - tam_ventana (int): Número de muestras que van a incluirse en cada uno de los segmentos en los que se dividen cada una de las señales
    - paso_solap (int): Número de muestras que se van a desplazar entre un segmento y el siguiente, es decir, el desplazamiento entre los distintos segmentos
    - lim_inf (float, optional): Límite inferior de frecuencia (Hz) que se va a incluir como parte de la señal filtrada
    - lim_sup (float, optional): Valor máximo de frecuencia (Hz) que va a pasar el filtro pasabanda aplicado a la señal
    - ord (int, optional): Orden del filtro de Butterworth aplicado a la señal


  Return
  ---------------
    - segmentos_diccionario (list): Lista aplanaza que contiene la señal normalizada de cada uno de los segmentos en que se han dividido las señales del diccionario


  Raises
  --------------
    - ValueError: Muestra un mensaje de error por pantalla en el caso de que algún objeto wfdb.Record no se haya podido procesar correctamente
  """

  segmentos_diccionario = []

  media, desviacion = obtener_param_globales (diccionario_registros)
  for nombre, record in diccionario_registros.items():
    try:
      segmentos_record = procesamiento_record_individual (record, tam_ventana, paso_solap, media, desviacion, lim_inf, lim_sup, ord)
      segmentos_diccionario.extend (segmentos_record)

    except Exception as e:
      raise ValueError (f"Se ha obtenido un error procesando el record {nombre}. Inténtalo de nuevo.")

  return segmentos_diccionario


In [60]:
a = procesar_multiples_registros (diccionario_señales, 300, 100)

print (a[0][0])

-0.27422559940025415


In [61]:
def descargar_segmentos_procesados (array_segmentos, ruta, nombre):

  """
  Realiza la descarga del array 2D con los segmentos normalizados en los que se han dividido las señales monocanales que integran el dataset.
  Este array presenta la forma [número_segmentos, tamaño_ventana], ya que todos estos segmentos van a presentar la misma longitud, igual al tamaño de ventana especificado a la
  hora de la creación de los mismos.

  La descarga se va realizar en Google Drive, en la ruta especificada como argumento de la función y bajo el nombre y extensión deseado por el usuario.

  Parámetros
  ----------------
    - array_segmentos (np.array): Array 1D que contiene todos los segmentos filtrados y normalizados en los que se han dividido las señales del dataset de trabajo

    - ruta (str): Ruta de Google Drive en la que se desea guardar dichos segmentos
      Ejemplo: '/content/drive/MyDrive/dataset_ECG/challenge2017/entrenamiento_GAN'

    - nombre (str): Nombre y extensión del archivo que se va a guardar en la ruta especificada
      Ejemplo: 'segmentos_ecg_procesados.npy'

  Return
  ----------------
    - None: Esta función no devuelve nada, ya que realiza el guardado de los segmentos en la dirección de Google Drive especificada

  Raises
  ----------------
    - ValueError: Se muestra un error por pantalla en el caso de que no sea posible realizar la descarga del array del dataset en la ruta o con el nombre especificado

  """

  from google.colab import drive                                         # Importa la biblioteca de Google Drive dentro de la función para poder conectarse a él
  drive.mount('/content/drive')                                          # Conecta el entorno de trabajo con Google Drive para poder almacenar los archivos pertinentes

  array_2d_segmentos = np.array(array_segmentos)                         # Convierte el array 1D en uno bidimensional del tipo [num_segmentos, tamaño_ventana]
  ruta_almacenamiento = ruta                                             # Crea una variable interna con la ruta en la que se quiere almacenar el array 2D creado

  os.makedirs (ruta_almacenamiento, exist_ok=True)                       # Crea el directorio especificado como argumento en el caso de que este no exista

  nombre_archivo = nombre                                                # Crea una variable interna con el nombre y extensión que se desea proporcionar al archivo con los segmentos

  ruta_completa = os.path.join (ruta_almacenamiento, nombre_archivo)     # Crea una variable con la ruta completa en la que se va a almacenar el archivo correspondiente

  try:                                                                   # Intenta realizar el guardado del array bidimensional en la ruta completa creada
    np.save (ruta_completa, array_2d_segmentos)
    print (f"Los segmentos procesados han sido guardados correctamente en la ruta {ruta_completa}.")   # Muestra un mensaje por pantalla en el caso de que consiga hacerlo

  except Exception as e:                                                 # Muestra un mensaje de error por pantalla en el caso de que no sea posible realizar la descarga
    raise ValueError ("Ha ocurrido un error y no se ha podido guardar correctamente el array que contiene los segmentos procesados. Inténtalo de nuevo.")





In [62]:
ruta_almacenamiento = "/content/drive/MyDrive/dataset_ECG/challenge2017/entrenamiento_GAN"

# El nombre del archivo debe presentar la extensión '.npy' ya que se trata de la extensión específica para objetos NumPy (como el array 2D en este caso),
# lo cual es útil para hacer posteriormente la carga en la red GAN

In [63]:
descargar_segmentos_procesados (a, ruta_almacenamiento, "descarga_prueba.npy")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Los segmentos procesados han sido guardados correctamente en la ruta /content/drive/MyDrive/dataset_ECG/challenge2017/entrenamiento_GAN/descarga_prueba.npy.
